---
title: AI Lean Mania
date: 2026-10-28
---

The world of AI is changing so fast, I have not equilibriated old ideas and projects. My way of going about things is based on an outdated perspective of what is easy and hard.

I like automated theorems provers. It is possible that AI is going to eat this category of things.

After a couple days of mania, instruction AI to write computer on my hom laptop at parties and such, my mania broke. I'm left with a sense of disgust with the work and with myself.

I like code staying small to a fault. Self contained tools. Little demo programs to explain ideas.

This is kind of cool though?  Defn -> ai_proof -> human




# Goto-Lean

In [31]:
%%file /tmp/sum.c
#include <stdio.h>
int sum(int n){
    int acc = 0;
    for(int i = 0; i < n; i++){
        acc += i;
    }
    return acc;
}

int main(){
    printf("sum 4 = %d", sum(4));
    return 0;
}

Overwriting /tmp/sum.c


In [32]:
! gcc /tmp/sum.c -o /tmp/sum && /tmp/sum

sum 4 = 6

In [16]:
! cbmc /tmp/sum.c --function sum

CBMC version 6.11.0 (cbmc-6.11.0) 64-bit x86_64 linux
Type-checking sum
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Starting Bounded Model Checking
^C


In [18]:
! goto-cc /tmp/sum.c -o /tmp/sum.gb

In [21]:
! ~/vibe_coding/goto-lean/build/bin/goto-lean /tmp/sum.gb --function sum -o /tmp/sum.lean

In [22]:
! cat /tmp/sum.lean

-- CBMC Running with 8 object bits, 56 offset bits (default)
-- Generated by goto-lean; edit proofs and harnesses in separate modules.
import GotoLean

namespace v_sum
open GotoLean

abbrev Result := BitVec 32

structure Locals where
  v_i : BitVec 32 := 0 -- sum::1::1::i
  v_acc : BitVec 32 := 0 -- sum::1::acc
  v_n : BitVec 32 := 0 -- sum::n
  result : Option Result := none
  deriving Repr

inductive PC where
  | b0 -- GOTO 0; ../../../../tmp/sum.c:2
  | b4 -- GOTO 4; ../../../../tmp/sum.c:3
  | b5 -- GOTO 5; ../../../../tmp/sum.c:4
  | b8 -- GOTO 8; ../../../../tmp/sum.c:3
  | b13 -- GOTO 13; ../../../../tmp/sum.c:7
  deriving Repr, BEq

abbrev Frame := PC × Locals

def step (frame : Frame) : Action (Next Frame Result) := do
  let (pc, initial) := frame
  let mut s := initial
  match pc with
  | .b0 =>
    -- sum: GOTO 0; ../../../../tmp/sum.c:2
    let t0 ← choose "sum: GOTO 0; ../../../../tmp/sum.c:2" 32
    s := { s with v_acc := t0 }
    -- sum: GOTO 1; ../../../../tmp/sum.c:2
 

In [38]:
%%file /tmp/runsum.lean
import .sum
#eval sum.eval 4 10000 []

Overwriting /tmp/runsum.lean


In [39]:
! cd /tmp && lean /tmp/runsum.lean

/tmp/runsum.lean:1:6: error: unexpected token '.'; expected identifier


Why have fuel? Why not just make it partial. 
Is it useful to return the choices list?
I think I might rather have the option to print the library inline.



# Yosys-Lean


In [41]:
%%file /tmp/counter.v
module counter(
    input clk,
    input [0:0] reset,
    output [15:0] out0
);
reg [15:0] count;
assign out0 = count;
always @(posedge clk) begin
    count <= ((reset == 1'd1) ? 16'd0 : (count + 16'd1));
end
endmodule

Overwriting /tmp/counter.v


In [45]:
! iverilog /tmp/counter.v  -o /tmp/counter && vvp /tmp/counter

In [43]:
! ebmc /tmp/counter.v

Converting
Type-checking Verilog::counter
Synthesis Verilog::counter
no properties


In [55]:
! ~/vibe_coding/yosys-lean/build/yosys/yosys -m ~/vibe_coding/yosys-lean/build/yosys_lean.so -p '\
  read_verilog -formal /tmp/counter.v;\
  hierarchy -check -top counter;\
  lean_prepare;\
  write_lean -named /tmp/Top.lean\
'


 /----------------------------------------------------------------------------\
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |  Copyright (C) 2012 - 2026  Claire Xenia Wolf <claire@yosyshq.com>         |
 |  Distributed under an ISC-like license, type "license" to see terms        |
 \----------------------------------------------------------------------------/
 Yosys 0.70+186 (git sha1 3718f7f13, Release, GNU /usr/bin/c++ 15.2.0)

-- Running command `   read_verilog -formal /tmp/counter.v;   hierarchy -check -top counter;   lean_prepare;   write_lean -named /tmp/Top.lean ' --

1. Executing Verilog-2005 frontend: /tmp/counter.v
Parsing formal Verilog input from `/tmp/counter.v' to AST representation.
Generating RTLIL representation for module `\counter'.
Successfully finished Verilog frontend.

2. Executing HIERARCHY pass (managing design hierarchy).

2.1. Analyzing design hierarchy..
Top module:  \counter

2.2. Analyzing design hierarchy..
Top modu

In [56]:
! cat /tmp/Top.lean

import YosysLean

-- Generated by yosys-lean; do not edit.
-- Yosys: Yosys 0.70+186 (git sha1 3718f7f13, Release, GNU /usr/bin/c++ 15.2.0)
-- Module: counter; source: /tmp/counter.v:1.1-11.10
-- One eval = one rising edge; outputs observe the pre-edge state.
-- Clock port omitted from Inputs: clk; src: /tmp/counter.v:2.11-2.14
namespace v_counter

structure Inputs where
  -- input reset; src: /tmp/counter.v:3.17-3.22
  v_reset : BitVec 1
  deriving DecidableEq, Repr, Inhabited

structure State where
  -- register $procdff$5; aliases: count out0; src: /tmp/counter.v:8.1-10.4
  v_count : BitVec 16
  deriving DecidableEq, Repr, Inhabited

structure Outputs where
  -- output out0; src: /tmp/counter.v:4.19-4.23
  v_out0 : BitVec 16
  deriving DecidableEq, Repr, Inhabited

-- Named intermediate expressions for local proof lemmas.
namespace Signals
set_option linter.unusedVariables false

-- out0 : state; src: /tmp/counter.v:4.19-4.23
def v_out0 (inputs : Inputs) (state : State) : BitVec 16 :

In [57]:
%%file /tmp/counter.lean
import Top

#eval eval {reset := 1} {count := 42}


Writing /tmp/counter.lean


We need a better way of executing this thing in my repl

In [58]:
! cd /tmp && lean /tmp/counter.lean

/tmp/counter.lean:1:0: error: unknown module prefix 'Top'

No directory 'Top' or file 'Top.olean' in the search path entries:
/home/philip/.elan/toolchains/leanprover--lean4---v4.35.0/lib/lean


# Pcode-Lean

